# 32 · Code RAG：让模型读得懂代码库

> “这个 Controller 调用了哪个 Service？”代码问答的关键不在“找相似行”，而在结构和依赖。

**本文件覆盖知识点**：Code Chunking / AST / Function·Class Retrieval / Dependency Retrieval / Call Graph / Repository RAG

In [1]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 代码切分：别按字数切

代码的天然单位是函数和类，而不是“1000 字符”硬切。

```text
差的切分: 把 def foo(...) 切成两半 → 语法断裂、无法理解
好的切分: 以函数/类为 chunk，可带 docstring/import
更好:    用 AST(抽象语法树)解析 → 精确定位每个函数体及其签名
```

In [2]:
# 用 Python 的 ast 演示“按函数切代码 + 提取调用关系”
import ast

code = '''
def get_user(id):
    return find_user(id)

def order_service(user_id):
    u = get_user(user_id)
    return create_order(u)
'''

tree = ast.parse(code)
funcs = {}   # 函数名 -> 它调用了谁(从函数体里找 Call 的名字)
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        calls = {n.func.id for n in ast.walk(node)
                 if isinstance(n, ast.Call) and isinstance(n.func, ast.Name)}
        funcs[node.name] = sorted(calls)

print('函数与调用关系(Call Graph):')
for f, calls in funcs.items():
    print(f'  {f} -> {calls}')
print('\n=> 有了这张图，就能回答“order_service 最终依赖谁”这类依赖检索问题。')

函数与调用关系(Call Graph):
  get_user -> ['find_user']
  order_service -> ['create_order', 'get_user']

=> 有了这张图，就能回答“order_service 最终依赖谁”这类依赖检索问题。


## 2. Repository RAG 的三层检索

```text
用户问: 这个 Controller 调用了哪个 Service？
  ① 定位: 检索到相关 Class/Function chunk      (Function Retrieval)
  ② 依赖: 沿 Call Graph 找它调用的 Service      (Dependency Retrieval)
  ③ 组织: 把调用链相关代码拼成上下文给 LLM
```

- **Function/Class Retrieval**：按签名+docstring+实现做索引；
- **Dependency Retrieval**：import/调用关系（AST 已能提取）；
- 代码与文档混合场景（代码+注释+README）最考验“结构化切分 + 图”。

In [4]:
# 知识点·真调说明：自然语言 → 检索动作 —— 让模型把一句开发问题翻译成“该检索什么代码符号”
import json as _json
out = _llm_live(
    prompt='代码库分层为 Controller/Service/Repository。用户问：“UserController 的登录接口调用了哪个 Service？”'
           '请把这个问题翻译成可执行的检索动作：先定位要检索的代码符号，再说拿到它之后要沿调用链追查什么。',
    system='你是 Code RAG 的检索规划器。输出一个 JSON，字段：'
           '{"locate": "<第一步检索的符号或检索词>", "dependency": "<拿到符号后要沿调用链追查的调用>", '
           '"plan": "<一句话说明>"}。只输出 JSON，禁止输出其它文字。',
    fallback='未配置 Key 的固定样例：\n'
             '{"locate": "UserController 类及其 login 方法", '
             '"dependency": "login 方法体内调用的 xxxService.xxx()", '
             '"plan": "先按 Controller 定位到 login 方法，再沿 Call Graph 找它调用的 Service 接口与实现"}',
    temperature=0.1,
)
if out is None:
    out = ('{"locate": "UserController 类及其 login 方法", '
           '"dependency": "login 方法体内调用的 xxxService.xxx()", '
           '"plan": "先定位到 login，再沿调用链找 Service"}')
    print('（以上为固定样例；下面用样例演示程序化解析检索计划）')
try:
    plan = _json.loads(out)
    print('解析成功 → 先检索「%s」' % plan['locate'])
    print('          再追查「%s」' % plan['dependency'])
except Exception as e:
    print('未解析成 JSON：', e, '—— 说明需在 prompt 里收紧输出格式。')
print('模型交出的不是“猜答案”，而是检索该从哪开始、下一步追什么。')
print('→ 这就是 Repository RAG 的 Function Retrieval(定位) + Dependency Retrieval(沿调用链找) 的可执行形态。')

—— 模型实时输出 ——
{"locate": "UserController.login", "dependency": "call", "plan": "定位 UserController 的 login 方法，沿其调用链向上查找直接调用的 Service 方法"}
解析成功 → 先检索「UserController.login」
          再追查「call」
模型交出的不是“猜答案”，而是检索该从哪开始、下一步追什么。
→ 这就是 Repository RAG 的 Function Retrieval(定位) + Dependency Retrieval(沿调用链找) 的可执行形态。


In [5]:
# 知识点·真调说明：代码语义描述 —— 让模型把一段无注释函数“翻译”成可检索的自然语言说明
_llm_live(
    prompt='下面是一段没有注释的函数源码。请用 2~3 句中文讲清它的职责与输入输出，'
           '作为给它做语义索引时用的描述文本：\n'
           'def order_service(user_id):\n'
           '    u = get_user(user_id)\n'
           '    return create_order(u)',
    system='你是代码库索引助手。只输出对该函数职责的中文说明，不写代码、不点评。',
    fallback='未配置 Key 的固定样例：\n'
             '该函数接收 user_id，先调用 get_user 取出用户信息，再用 create_order 为该用户创建订单并返回结果。'
             '它把“查用户 + 下单”两个能力编排成“为用户下单”这一业务动作。',
    temperature=0.2,
)
print('有了这段语义描述，用户问“怎么给用户下单 / 下单接口在哪”这类自然语言，就能命中这个函数。')
print('→ 函数级 chunk + docstring/语义描述，是代码库能被语义检索的前提；代码 embedding 学不到的调用意图靠它补齐。')

—— 模型实时输出 ——
该函数根据用户ID查询用户信息，并基于该用户创建一个新订单。  
输入为用户唯一标识符user_id，输出为创建成功的订单对象。  
其核心职责是封装“查用户→下订单”的业务流程。
有了这段语义描述，用户问“怎么给用户下单 / 下单接口在哪”这类自然语言，就能命中这个函数。
→ 函数级 chunk + docstring/语义描述，是代码库能被语义检索的前提；代码 embedding 学不到的调用意图靠它补齐。


## 3. 代码检索的进阶

- 代码专用 embedding（如 code embedding）常比通用模型更能区分语义相近的函数；
- 与 GraphRAG 结合：把“函数→函数”依赖建成图，做多跳；
- 大仓库先做“文件/模块级路由”，再进文件细搜。

## 小结

- Code RAG 切分以函数和类为单位，用 AST 拿结构；
- 关键武器是 Call Graph（谁调谁），支撑依赖和多跳检索；
- 目标从“语义相近”升级为“结构正确”。